# Retrieval Sanity Check

Same query → all three collections → compare top-3 chunks.

Goal: confirm each chunking strategy returns meaningfully different results, and that scores and source files look reasonable.

In [1]:
import sys
sys.path.insert(0, "..")

from src.embedding import Embedder
from src.vectorstore import VectorStore
from configs.settings import AppConfig

cfg      = AppConfig()
embedder = Embedder()
store    = VectorStore()

COLLECTIONS = {
    "fixed":     cfg.collection_fixed,
    "recursive": cfg.collection_recursive,
    "semantic":  cfg.collection_semantic,
}
print("collections:", list(COLLECTIONS.values()))

/Users/harsh/drive/workspace/ai_projects/doc-rag-engine/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


collections: ['rag_fixed', 'rag_recursive', 'rag_semantic']


## Embed the query

One embedding call — reused across all three collections.

In [2]:
QUERY = "How does retrieval augmented generation reduce hallucinations in large language models?"

q_dense  = embedder.embed_dense([QUERY])[0].tolist()
q_sparse = embedder.embed_sparse([QUERY])[0]

print(f"query  : {QUERY}")
print(f"dense  : {len(q_dense)}-dim vector")
print(f"sparse : {len(q_sparse.indices)} non-zero SPLADE terms")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.76it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]

query  : How does retrieval augmented generation reduce hallucinations in large language models?
dense  : 384-dim vector
sparse : 50 non-zero SPLADE terms


## Dense search — top 3 per collection

In [3]:
import textwrap

def show_hits(hits, strategy, search_type):
    print(f"\n{'━'*70}")
    print(f"  [{strategy.upper()}]  {search_type}  —  query: {QUERY[:55]}...")
    print(f"{'━'*70}")
    for i, h in enumerate(hits):
        print(f"\n  rank {i+1}  score={h.score:.4f}  source={h.filename}  "
              f"strategy={h.chunk_strategy}  chunk_idx={h.chunk_index}")
        print(textwrap.fill(h.text.strip(), width=80,
                            initial_indent="  ", subsequent_indent="  "))

for strategy, collection in COLLECTIONS.items():
    hits = store.search_dense(q_dense, collection, limit=3)
    show_hits(hits, strategy, "DENSE")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [FIXED]  DENSE  —  query: How does retrieval augmented generation reduce hallucin...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  rank 1  score=0.7888  source=corrective_rag.pdf  strategy=fixed  chunk_idx=0
  Corrective Retrieval Augmented Generation  Shi-Qi Yan 1 * , Jia-Chen Gu 2 * ,
  Yun Zhu 3 , Zhen-Hua Ling 1  1 National Engineering Research Center of Speech
  and Language Information Processing, University of Science and Technology of
  China, Hefei, China 2 Department of Computer Science, University of
  California, Los Angeles 3 Google DeepMind yansiki@mail.ustc.edu.cn ,
  gujc@ucla.edu , yunzhu@google.com , zhling@ustc.edu.cn  Abstract  Large
  language models (LLMs) inevitably exhibit hallucinations since the acc

  rank 2  score=0.7551  source=corrective_rag.pdf  strategy=fixed  chunk_idx=1
  ge models (LLMs) inevitably exhibit hallucinations since the accuracy of
  genera

## Sparse search — top 3 per collection

In [4]:
for strategy, collection in COLLECTIONS.items():
    hits = store.search_sparse(q_sparse, collection, limit=3)
    show_hits(hits, strategy, "SPARSE")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [FIXED]  SPARSE  —  query: How does retrieval augmented generation reduce hallucin...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  rank 1  score=26.7020  source=rag_hallucination.pdf  strategy=fixed  chunk_idx=216
  5 Conclusion  In this work, we have studied the problem of knowledge
  hallucination in conversational agents, an important problem as current
  systems often produce factually inaccurate generations. We have shown that
  this problem occurs independently of language model size or training data.
  Retrieval-augmented generation in particular is an intuitively promising
  solution to this problem, and in detailed experiments we have shown that this
  class of approaches significantly reduces the hallucination problem in

  rank 2  score=26.5467  source=rag_llm.pdf  strategy=fixed  chunk_idx=1
  tract -Large Language Models (LLMs) showcase impressive capabilities but
  encou

## Hybrid search (RRF) — top 3 per collection

This is the production search path — dense + sparse fused via Reciprocal Rank Fusion.

In [5]:
for strategy, collection in COLLECTIONS.items():
    hits = store.search_hybrid(q_dense, q_sparse, collection, limit=3)
    show_hits(hits, strategy, "HYBRID")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [FIXED]  HYBRID  —  query: How does retrieval augmented generation reduce hallucin...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  rank 1  score=0.7500  source=rag_hallucination.pdf  strategy=fixed  chunk_idx=216
  5 Conclusion  In this work, we have studied the problem of knowledge
  hallucination in conversational agents, an important problem as current
  systems often produce factually inaccurate generations. We have shown that
  this problem occurs independently of language model size or training data.
  Retrieval-augmented generation in particular is an intuitively promising
  solution to this problem, and in detailed experiments we have shown that this
  class of approaches significantly reduces the hallucination problem in

  rank 2  score=0.7000  source=corrective_rag.pdf  strategy=fixed  chunk_idx=0
  Corrective Retrieval Augmented Generation  Shi-Qi Yan 1 * , Jia-Chen Gu 2 *

## Score comparison table

Side-by-side scores for the top hit from each strategy and search type.

In [6]:
print(f"\n{'─'*78}")
print(f"  {'STRATEGY':<12} {'SEARCH':<8} {'SCORE':>6}  {'SOURCE':<30}  CHUNK_IDX")
print(f"{'─'*78}")

for strategy, collection in COLLECTIONS.items():
    for search_type, fn in [
        ("dense",  lambda c: store.search_dense(q_dense, c, limit=1)),
        ("sparse", lambda c: store.search_sparse(q_sparse, c, limit=1)),
        ("hybrid", lambda c: store.search_hybrid(q_dense, q_sparse, c, limit=1)),
    ]:
        hits = fn(collection)
        if hits:
            h = hits[0]
            print(f"  {strategy:<12} {search_type:<8} {h.score:>6.4f}  {h.filename:<30}  {h.chunk_index}")

print(f"{'─'*78}")


──────────────────────────────────────────────────────────────────────────────
  STRATEGY     SEARCH    SCORE  SOURCE                          CHUNK_IDX
──────────────────────────────────────────────────────────────────────────────
  fixed        dense    0.7888  corrective_rag.pdf              0
  fixed        sparse   26.7020  rag_hallucination.pdf           216
  fixed        hybrid   0.7500  rag_hallucination.pdf           216
  recursive    dense    0.8021  corrective_rag.pdf              1
  recursive    sparse   28.6490  corrective_rag.pdf              1
  recursive    hybrid   1.0000  corrective_rag.pdf              1
  semantic     dense    0.8002  corrective_rag.pdf              1
  semantic     sparse   28.7990  corrective_rag.pdf              1
  semantic     hybrid   1.0000  corrective_rag.pdf              1
──────────────────────────────────────────────────────────────────────────────


## Second query — filter by source file

Verify metadata filtering works: restrict to arXiv papers only.

In [7]:
QUERY2 = "What evaluation metrics are used for RAG systems?"
q2_dense  = embedder.embed_dense([QUERY2])[0].tolist()
q2_sparse = embedder.embed_sparse([QUERY2])[0]

print(f"query: {QUERY2}\n")
for strategy, collection in COLLECTIONS.items():
    hits = store.search_hybrid(
        q2_dense, q2_sparse, collection, limit=3,
        filters={"source_file": "ragas.pdf"},
    )
    print(f"[{strategy}] filtered to ragas.pdf → {len(hits)} hits")
    for h in hits:
        print(f"  score={h.score:.4f}  {h.filename}  chunk={h.chunk_index}  {h.text[:80]!r}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

query: What evaluation metrics are used for RAG systems?

[fixed] filtered to ragas.pdf → 3 hits
  score=0.6667  ragas.pdf  chunk=20  'he probability of generating the reference given the generated answer).\n\n3 Evalu'
  score=0.6667  ragas.pdf  chunk=2  'he retrieval system to identify relevant and focused context passages, the abili'
  score=0.5667  ragas.pdf  chunk=9  'ent Ragas 1 , a framework for the automated assessment\n\n1 Ragas is available at '

[recursive] filtered to ragas.pdf → 3 hits
  score=1.0000  ragas.pdf  chunk=2  '. Evaluating RAG architectures is, however, challenging because there are severa'
  score=0.4500  ragas.pdf  chunk=58  'We have highlighted the need for automated reference-free evaluation of RAG syst'
  score=0.4444  ragas.pdf  chunk=55  'The results in Table 1 show that our proposed metrics are much closer aligned wi'

[semantic] filtered to ragas.pdf → 3 hits
  score=1.0000  ragas.pdf  chunk=2  '. Evaluating RAG architectures is, however, challenging be